In [1]:
import os
import pandas as pd
import sqlite3

In [2]:
DB_PATH = r"C:\Users\sveto\OneDrive\Desktop\eko_inv\eko_cards.db"
BASE_DIR = r"C:\Users\sveto\OneDrive\Desktop\eko_inv"
PRICES_PATH = r"C:\Users\sveto\OneDrive\Desktop\eko_inv\data\prices.xlsx"

In [3]:
def transaction_import():

    # -------------------------
    # READ EXCEL
    # -------------------------
    data = pd.read_excel(
        os.path.join(BASE_DIR, "data", "eko_transaction.xlsx"),
        dtype={"Number": str}
    )

    # -------------------------
    # CLEAN COLUMN NAMES
    # -------------------------
    data.columns = data.columns.str.strip()

    # -------------------------
    # CLEAN MATERIAL
    # -------------------------
    data["Material"] = (
        data["Material"]
        .astype(str)
        .str.replace(r'^\d+\s*', '', regex=True)   # remove product code
        .str.replace(r"\s+", " ", regex=True)      # normalize spaces
        .str.strip()
        .str.upper()                               # normalize text
    )

    # -------------------------
    # CLEAN VEHICLE
    # -------------------------
    data["Name"] = (
        data["Name"]
        .astype(str)
        .str.replace(" ", "", regex=False)
        .str.strip()
    )

    # -------------------------
    # SELECT COLUMNS
    # -------------------------
    data = data[
        [
            "Plant",
            "Name",
            "Number",
            "Material",
            "Date",
            "Bill.qty",
            "Bill.qty.1",
            "FinPr",
            "Tot Amount",
            "Auth.time",
            "Km stand"
        ]
    ]

    # -------------------------
    # FORMAT DATE
    # -------------------------
    data["Date"] = pd.to_datetime(
        data["Date"],
        dayfirst=True,
        errors="coerce"
    )

    data = data[data["Date"].notna()]
    data["Date"] = data["Date"].dt.strftime("%Y-%m-%d")

    # -------------------------
    # CLEAN CARD NUMBERS
    # -------------------------
    data["Number"] = (
        data["Number"]
        .astype(str)
        .str.replace(r"\D", "", regex=True)
    )

    data = data[data["Number"].str.len() > 0]

    # -------------------------
    # REMOVE NON TRANSACTIONS
    # -------------------------
    data["Bill.qty"] = pd.to_numeric(data["Bill.qty"], errors="coerce")
    data = data[data["Bill.qty"] > 0]

    # -------------------------
    # RENAME COLUMNS
    # -------------------------
    data = data.rename(columns={
        "Plant": "plant",
        "Name": "name",
        "Number": "card_number",
        "Material": "material",
        "Date": "date",
        "Bill.qty": "bill_qty",
        "Bill.qty.1": "bill_qty2",
        "FinPr": "price",
        "Tot Amount": "amount",
        "Auth.time": "auth_time",
        "Km stand": "Km_stand"
    })
    data.columns = data.columns.str.strip()
    # -------------------------
    # CONNECT DATABASE
    # -------------------------
    conn = sqlite3.connect(DB_PATH)

    data.to_sql(
        "transactions",
        conn,
        if_exists="append",
        index=False
    )

    conn.close()

    print(f"Transactions imported successfully: {len(data)} rows")

In [4]:
def price_import():

    # -------------------------
    # READ EXCEL
    # -------------------------
    prices = pd.read_excel(PRICES_PATH)

    # -------------------------
    # CLEAN COLUMN NAMES
    # -------------------------
    prices.columns = prices.columns.str.strip()

    # -------------------------
    # SELECT COLUMNS
    # -------------------------
    prices = prices[["ДАТА", "ФИРМА", "ПРОДУКТ", "КРАЙНА_ЦЕНА"]]

    # -------------------------
    # RENAME COLUMNS
    # -------------------------
    prices = prices.rename(columns={
        "ДАТА": "date",
        "ФИРМА": "company",
        "ПРОДУКТ": "product",
        "КРАЙНА_ЦЕНА": "final_price"
    })

    # -------------------------
    # CLEAN COMPANY
    # -------------------------
    prices["company"] = (
        prices["company"]
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    # -------------------------
    # CLEAN PRODUCT
    # -------------------------
    prices["product"] = (
        prices["product"]
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.upper()
    )

    # -------------------------
    # FORMAT DATE
    # -------------------------
    prices["date"] = pd.to_datetime(
        prices["date"],
        dayfirst=True,
        errors="coerce"
    )

    prices = prices[prices["date"].notna()]
    prices["date"] = prices["date"].dt.strftime("%Y-%m-%d")

    # -------------------------
    # PRICE FORMAT
    # -------------------------
    prices["final_price"] = pd.to_numeric(
        prices["final_price"],
        errors="coerce"
    )

    prices = prices[prices["final_price"].notna()]

    # -------------------------
    # CONNECT DATABASE
    # -------------------------
    conn = sqlite3.connect(DB_PATH)

    # -------------------------
    # WRITE TO DATABASE
    # -------------------------
    prices.to_sql(
        "prices",
        conn,
        if_exists="append",
        index=False
    )

    conn.close()

    print(f"Prices imported successfully: {len(prices)} rows")

In [5]:
def company_import():

    cards = pd.read_excel("../data/cards.xlsx", dtype={"Number": str})

    cards = cards.rename(columns={
        "Company": "company",
        "Name": "vehicle",
        "Number": "card_number"
    })

    conn = sqlite3.connect(DB_PATH)

    cards.to_sql(
        "cards",
        conn,
        if_exists="replace",
        index=False
    )

    conn.close()

In [6]:
# def merge_tables():
#
#     conn = sqlite3.connect(DB_PATH)
#
#     query = """
#
#     SELECT
#         t.*,
#         c.company,
#         ROUND(p.final_price, 2) AS final_price
#
#     FROM transactions t
#
#     LEFT JOIN cards c
#         ON t.card_number = c.card_number
#
#     LEFT JOIN prices p
#         ON (
#             CASE
#                 WHEN INSTR(c.company, '%') > 0
#                 THEN TRIM(SUBSTR(c.company, 1, INSTR(c.company, '%') - 1))
#                 ELSE c.company
#             END
#         ) = p.company
#         AND UPPER(p.product) = UPPER(t.material)
#         AND p.date = (
#             SELECT MAX(p2.date)
#             FROM prices p2
#             WHERE p2.company = (
#                 CASE
#                     WHEN INSTR(c.company, '%') > 0
#                     THEN TRIM(SUBSTR(c.company, 1, INSTR(c.company, '%') - 1))
#                     ELSE c.company
#                 END
#             )
#             AND UPPER(p2.product) = UPPER(t.material)
#             AND p2.date <= t.date
#         )
#
#     """
#
#     data = pd.read_sql(query, conn)
#
#     conn.close()
#
#     return data

def merge_tables():

    conn = sqlite3.connect(DB_PATH)

    query = """

    SELECT
        t.*,
        c.company,
        ROUND(p.final_price, 2) AS final_price

    FROM transactions t

    LEFT JOIN cards c
        ON t.card_number = c.card_number

    LEFT JOIN prices p
        ON UPPER(c.company) LIKE '%' || UPPER(p.company) || '%'
        AND UPPER(p.product) = UPPER(t.material)
        AND p.date = (
            SELECT MAX(p2.date)
            FROM prices p2
            WHERE UPPER(c.company) LIKE '%' || UPPER(p2.company) || '%'
            AND UPPER(p2.product) = UPPER(t.material)
            AND p2.date <= t.date
        )

    """

    data = pd.read_sql(query, conn)

    conn.close()

    return data

In [7]:
def generate_result(data):

    os.makedirs(os.path.join(BASE_DIR, "result"), exist_ok=True)

    output_file = os.path.join(BASE_DIR, "result", "eko_transaction.xlsx")

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

        for company, df in data.groupby("company", dropna=False):

            company_name = "Unknown" if pd.isna(company) else str(company)
            sheet_name = company_name[:31]

            df["price"] = pd.to_numeric(df["price"], errors="coerce").round(2)
            df["final_price"] = pd.to_numeric(df["final_price"], errors="coerce").round(2)

            df["eko_total"] = (df["bill_qty"] * df["price"]).round(2)
            df["gta_total"] = (df["bill_qty"] * df["final_price"]).round(2)

            df = df[
                [
                    "date",
                    "plant",
                    "name",
                    "card_number",
                    "material",
                    "bill_qty",
                    "bill_qty2",
                    "final_price",
                    "gta_total",
                    "price",
                    "eko_total",
                    "auth_time",
                    "Km_stand"
                ]
            ]

            df = df.rename(columns={
                "date": "Дата",
                "plant": "Станция",
                "name": "Име",
                "card_number": "Номер на карта",
                "material": "Име на артикул",
                "bill_qty": "Литри",
                "bill_qty2": "Тип количество",
                "final_price": "GTA цена",
                "gta_total": "Сума по GTA цена",
                "price": "ЕКО цена",
                "eko_total": "Сума по ЕКО цена",
                "auth_time": "Час",
                "Km_stand": "Километри"
            })

            df.to_excel(writer, sheet_name=sheet_name, index=False)

    print("Output file generated:", output_file)

In [8]:
def delete_transactions():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("DELETE FROM transactions")

    conn.commit()
    conn.close()

In [9]:
def delete_company():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("DELETE FROM cards")

    conn.commit()
    conn.close()

In [10]:
def delete_prices():

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("DELETE FROM prices")

    conn.commit()
    conn.close()

    print("All prices deleted.")

In [11]:
def delete_prices_by_month(year, month):

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    month_str = f"{year}-{str(month).zfill(2)}"

    cursor.execute(
        """
        DELETE FROM prices
        WHERE strftime('%Y-%m', date) = ?
        """,
        (month_str,)
    )

    conn.commit()
    conn.close()

    print(f"Prices deleted for: {month_str}")

In [12]:
def execute_pipeline():
    delete_transactions()

    transaction_import()

    data = merge_tables()

    generate_result(data)